# multilingual-e5-large — Improved Deep Embedded Clustering (IDEC) (deep clustering, part 4/4)

IDEC is DEC (`clustering_e5_DEC.ipynb`) with one change: it **keeps the decoder and its
reconstruction loss during fine-tuning**, instead of discarding the decoder after pretraining.
The fine-tuning objective becomes `KL(P‖Q) + γ·reconstruction_loss` (γ=0.1 here), jointly
optimizing the full autoencoder plus the cluster centers.

Motivation (from the IDEC literature): DEC\'s pure clustering objective can distort the latent
space\'s local structure as it sharpens cluster assignments, since nothing during fine-tuning
requires the latent codes to still be reconstructable. Keeping the reconstruction loss acts as a
regularizer against that distortion. Comparing this notebook\'s scores against
`clustering_e5_DEC.ipynb`\'s is a direct, on-this-corpus test of whether that regularization
actually helps here.

In [ ]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)
torch_seed_set = False  # set after torch import below

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_idec")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

In [ ]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

In [ ]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

In [ ]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

In [ ]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

In [ ]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

In [ ]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

In [ ]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

In [ ]:
# Embed with multilingual-e5-large (the finalized embedding model for this project),
# mean pooling — established as the best strategy for this model on this corpus
import torch
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

In [ ]:
# Standardize before feeding the autoencoder (zero mean / unit variance per dimension) —
# standard practice for deep clustering methods, keeps every input dimension on a comparable
# scale for the reconstruction loss even though the raw embeddings are already L2-normalized
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(embeddings).astype(np.float32)
X_tensor = torch.tensor(X, dtype=torch.float32)
print(X.shape)

In [ ]:
# Shared autoencoder architecture used by all four deep-clustering notebooks
import torch.nn as nn

INPUT_DIM = X.shape[1]
HIDDEN_DIMS = [256, 64]
LATENT_DIM = 16


class Autoencoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dims=HIDDEN_DIMS, latent_dim=LATENT_DIM):
        super().__init__()
        enc_layers, prev = [], input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers, prev = [], latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

In [ ]:
# Step 1: pretrain the autoencoder on reconstruction loss only
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
autoencoder = Autoencoder().to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_tensor), batch_size=256, shuffle=True)

PRETRAIN_EPOCHS = 50
for epoch in range(PRETRAIN_EPOCHS):
    total_loss = 0.0
    for (batch,) in loader:
        batch = batch.to(device)
        z, x_hat = autoencoder(batch)
        loss = nn.functional.mse_loss(x_hat, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"pretrain epoch {epoch + 1:>3}/{PRETRAIN_EPOCHS}  MSE: {total_loss / len(X_tensor):.4f}")

In [ ]:
# Step 2: pick k and initialize cluster centers by running K-Means on the pretrained latent
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

autoencoder.eval()
with torch.no_grad():
    pretrained_latent, _ = autoencoder(X_tensor.to(device))
    pretrained_latent = pretrained_latent.cpu().numpy()

candidate_k = list(range(10, 100, 15)) + list(range(100, 700, 100))
sweep_results = []
for k in candidate_k:
    if k >= len(pretrained_latent):
        continue
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(pretrained_latent)
    sil = silhouette_score(pretrained_latent, labels, metric="cosine")
    sweep_results.append((k, sil))

sweep_df = pd.DataFrame(sweep_results, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)
best_k = int(sweep_df.iloc[0]["k"])
print(f"Best k (from pretrained-latent K-Means sweep): {best_k}")

kmeans_init = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit(pretrained_latent)
cluster_centers = nn.Parameter(
    torch.tensor(kmeans_init.cluster_centers_, dtype=torch.float32, device=device)
)

In [ ]:
# Step 3: same soft-assignment / target-distribution machinery as DEC
def soft_assign(z, centers, alpha=1.0):
    dist_sq = torch.cdist(z, centers) ** 2
    numerator = (1.0 + dist_sq / alpha) ** (-(alpha + 1.0) / 2.0)
    return numerator / numerator.sum(dim=1, keepdim=True)


def target_distribution(q):
    weight = (q ** 2) / q.sum(dim=0)
    return (weight.t() / weight.sum(dim=1)).t()

In [ ]:
# Step 4: fine-tune the FULL autoencoder (encoder + decoder) + cluster centers, minimizing
# KL(P || Q) + gamma * reconstruction_loss jointly. This is IDEC\'s key difference from DEC: the
# reconstruction term is kept during fine-tuning (not just pretraining) specifically to stop the
# clustering objective from distorting the latent space\'s local structure — DEC drops the decoder
# entirely at this stage, IDEC keeps it as a regularizer.
GAMMA_RECONSTRUCTION = 0.1
RETARGET_EVERY = 5
N_ROUNDS = 10  # -> RETARGET_EVERY * N_ROUNDS = 50 total epochs
optimizer = torch.optim.Adam(list(autoencoder.parameters()) + [cluster_centers], lr=1e-4)

for round_idx in range(N_ROUNDS):
    with torch.no_grad():
        z = autoencoder.encoder(X_tensor.to(device))
        q = soft_assign(z, cluster_centers)
        p = target_distribution(q)

    for epoch in range(RETARGET_EVERY):
        z, x_hat = autoencoder(X_tensor.to(device))
        q = soft_assign(z, cluster_centers)
        kl_loss = nn.functional.kl_div(q.log(), p, reduction="batchmean")
        recon_loss = nn.functional.mse_loss(x_hat, X_tensor.to(device))
        loss = kl_loss + GAMMA_RECONSTRUCTION * recon_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"round {round_idx + 1:>2}/{N_ROUNDS}  KL(P||Q)={kl_loss.item():.4f}  recon={recon_loss.item():.4f}")

In [ ]:
autoencoder.eval()
with torch.no_grad():
    final_latent = autoencoder.encoder(X_tensor.to(device))
    q = soft_assign(final_latent, cluster_centers)
    final_labels = q.argmax(dim=1).cpu().numpy()
    final_latent = final_latent.cpu().numpy()

print(pd.Series(final_labels).value_counts().head(20))
print("Number of non-empty clusters:", len(set(final_labels)))

In [ ]:
# Clustering evaluation (cosine metric, matching every other notebook in this project)
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity

n_clusters = len(set(final_labels))
sil = silhouette_score(final_latent, final_labels, metric="cosine")
dbi = davies_bouldin_score(final_latent, final_labels)
ch = calinski_harabasz_score(final_latent, final_labels)

print(f"Model: {embedding_model_name} + IDEC")
print(f"Articles: {len(df)} | Clusters: {n_clusters} | Noise ratio: 0.00% (n/a for this algorithm)")
print(f"Silhouette Score (cosine): {sil:.4f}")
print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")

# Separation score computed on the raw e5 embeddings, for comparability with every other
# notebook in this project (not the AE latent space, which is method-specific)
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sims = cosine_similarity(embeddings[sample_idx])
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

scores_path = RESULTS_DIR / "e5_idec_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "IDEC",
    "embedding_dim": embeddings.shape[1],
    "latent_dim": LATENT_DIM,
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": 0.0,
    "silhouette": round(sil, 4),
    "davies_bouldin": round(dbi, 4),
    "calinski_harabasz": round(ch, 2),
}
row.update({"k": best_k, "gamma_reconstruction": GAMMA_RECONSTRUCTION})
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

In [ ]:
# Inspect sample titles per cluster
df["cluster_id"] = final_labels
for cluster_id, group in list(df.groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

In [ ]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_idec_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")